<a href="https://colab.research.google.com/github/Tunz10/Tunz10/blob/main/medicalMRI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import gradio
import transformers
import medpy
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

print("Gradio:", gradio.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "GPU bağlı değil")
print("Gerekli importlar başarılı ✅")

Gradio: 5.49.1
Transformers: 4.51.3
GPU: Tesla T4
Gerekli importlar başarılı ✅


In [ ]:
import sys
import subprocess
from pathlib import Path

import torch
import numpy as np
from PIL import Image

print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "GPU yok — T4 seçmelisin")

REPO = Path("/content/brain-segmentation-pytorch")

if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/mateuszbuda/brain-segmentation-pytorch.git",
        str(REPO)
    ], check=True)

sys.path.insert(0, str(REPO))

from dataset import BrainSegmentationDataset

# Küçük segmentasyon modeli CPU'da çalışır.
# GPU belleğini Qwen için ayırıyoruz.
unet = torch.hub.load(
    str(REPO),
    "unet",
    source="local",
    pretrained=True,
    in_channels=3,
    out_channels=1,
    init_features=32
).cpu().eval()

print("U-Net hazır.")

GPU: Tesla T4
Downloading: "https://github.com/mateuszbuda/brain-segmentation-pytorch/releases/download/v1.0/unet-e012d006.pt" to /root/.cache/torch/hub/checkpoints/unet-e012d006.pt
U-Net hazır.


In [ ]:
import kagglehub
from pathlib import Path

DATA_DIR = Path(
    kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")
)

case_folders = sorted({
    p.parent
    for p in DATA_DIR.rglob("*.tif")
    if "_mask" not in p.stem
})

if not case_folders:
    raise RuntimeError("Veri klasöründe MRI TIFF dosyası bulunamadı.")

cases = {
    f"{i} | {folder.name}": folder
    for i, folder in enumerate(case_folders)
}

print("Veri hazır.")
print("Vaka sayısı:", len(cases))
print("İlk vaka:", next(iter(cases)))

100%|██████████| 714M/714M [00:42<00:00, 17.4MB/s]

Extracting files...


Veri hazır.
Vaka sayısı: 220
İlk vaka: 0 | TCGA_CS_4941_19960909


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=2)
def load_case(case_name):
    # Orijinal modelin veri okuma ve ön işleme kodu.
    dataset = BrainSegmentationDataset(
        images_dir=str(cases[case_name]),
        subset="all",
        random_sampling=False,
        image_size=256
    )

    if not dataset.volumes:
        raise ValueError("Bu vaka yüklenemedi.")

    volume, reference = dataset.volumes[0]
    volume = volume.astype(np.float32)
    reference = np.asarray(reference)

    if reference.ndim == 4:
        reference = reference[..., 0]

    reference = reference > 0.5

    if not np.isfinite(volume).all():
        raise ValueError("MRI normalizasyonunda geçersiz değer oluştu.")

    predictions = []

    with torch.inference_mode():
        for start in range(0, len(volume), 8):
            batch = volume[start:start + 8].transpose(0, 3, 1, 2)
            batch = torch.from_numpy(np.ascontiguousarray(batch))
            predictions.append(unet(batch)[:, 0].numpy())

    probabilities = np.concatenate(predictions)

    return volume, reference, probabilities


def overlap_scores(prediction, reference):
    intersection = int((prediction & reference).sum())
    total = int(prediction.sum() + reference.sum())
    union = int((prediction | reference).sum())

    return {
        "Dice": 2 * intersection / total if total else 1.0,
        "IoU": intersection / union if union else 1.0,
        "Both masks empty": total == 0
    }


def segment(case_name, slice_index, threshold):
    volume, reference, probabilities = load_case(case_name)

    index = int(slice_index)

    if not 0 <= index < len(volume):
        raise ValueError(f"Kesit numarası 0–{len(volume)-1} arasında olmalı.")

    prediction = probabilities[index] >= threshold

    # FLAIR görüntüsünü ekranda göstermek için normalize et.
    gray = volume[index, ..., 1]
    low, high = np.percentile(gray, [1, 99])
    gray = np.clip((gray - low) / max(float(high - low), 1e-6), 0, 1)

    image = Image.fromarray((gray * 255).astype(np.uint8)).convert("RGB")

    overlay = np.array(image).astype(float)
    overlay[prediction] = (
        0.6 * overlay[prediction] + 0.4 * np.array([255, 40, 40])
    )
    overlay = Image.fromarray(overlay.astype(np.uint8))

    predicted_mask = Image.fromarray(
        (prediction.astype(np.uint8) * 255)
    )

    reference_mask = Image.fromarray(
        (reference[index].astype(np.uint8) * 255)
    )

    measurements = {
        "Processed slice": index,
        "Total processed slices": len(volume),
        "Threshold": float(threshold),
        "Selected pixels": int(prediction.sum()),
        "Processed grid area (%)": round(float(prediction.mean() * 100), 3),
        **overlap_scores(prediction, reference[index]),
        "Evaluation": "Same-source demo; not independent test accuracy.",
        "Units": "Processed grid pixels, not mm or tumor volume."
    }

    return image, overlay, predicted_mask, reference_mask, measurements


# İlk vakayı gerçekten yükleyerek kontrol et.
first_case = next(iter(cases))
volume, reference, probabilities = load_case(first_case)

print("İlk vaka başarıyla işlendi.")
print("Kesit sayısı:", len(volume))
print("MRI boyutu:", volume.shape)

reading all images...
preprocessing all volumes...
cropping all volumes...
padding all volumes...
resizing all volumes...
normalizing all volumes...
done creating all dataset
İlk vaka başarıyla işlendi.
Kesit sayısı: 21
MRI boyutu: (21, 256, 256, 3)


In [ ]:
import json
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

qwen = None
processor = None

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

def load_qwen():
    global qwen, processor

    if qwen is not None:
        return

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Qwen için GPU gerekiyor. Colab çalışma zamanı ayarından T4 seç."
        )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=256 * 28 * 28,
        max_pixels=512 * 28 * 28
    )

    qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        attn_implementation="sdpa"
    ).eval()


def generate_report(case_name, slice_index, threshold, language):
    image, overlay, _, _, measurements = segment(
        case_name, slice_index, threshold
    )

    load_qwen()

    # Referans maske ve Dice/IoU Qwen'e verilmez.
    evidence = {
        key: measurements[key]
        for key in [
            "Processed slice",
            "Total processed slices",
            "Threshold",
            "Selected pixels",
            "Processed grid area (%)",
            "Units"
        ]
    }

    prompt = f"""
Write a concise educational clinician-review draft in {language}.

Image 1: one processed brain MRI slice, displayed using FLAIR.
Image 2: the same slice with a predicted U-Net mask in red.
The red area is a software overlay, not actual tissue color.

Computed measurements:
{json.dumps(evidence)}

Use these sections:
1. Observations
2. Tentative interpretation
3. Points for the doctor to review
4. Limitations

Rules:
- Distinguish visible observations from algorithm measurements.
- A predicted abnormality is not proof of a tumor or cancer.
- If evidence is insufficient, explicitly abstain from diagnosis.
- Do not invent symptoms, history, tumor grade, histology,
  anatomical laterality, physical dimensions or confidence percentages.
- Do not prescribe medication, doses, surgery or treatment.
- Suggest reviewing the original full MRI study, other sequences,
  clinical history and a radiologist's interpretation.
- An empty mask does not rule out disease.
- This general-purpose vision-language model is not clinically validated.
"""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image, overlay],
        padding=True,
        return_tensors="pt"
    ).to(qwen.device)

    with torch.inference_mode():
        output = qwen.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    new_tokens = output[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    return (
        "UNVERIFIED AI DRAFT — Educational use only. "
        "Requires clinician review; not a diagnosis.\n\n"
        + answer
    )

print("Qwen bağlantı kodu hazır.")

Qwen bağlantı kodu hazır.


In [ ]:
import gradio as gr

def select_case(case_name):
    volume, _, _ = load_case(case_name)

    return (
        gr.update(
            minimum=0,
            maximum=len(volume) - 1,
            value=len(volume) // 2
        ),
        None, None, None, None, None, ""
    )


with gr.Blocks(title="MRI Medical Assistant") as demo:
    gr.Markdown("""
    # MRI Medical Assistant
    **U-Net segmentation + Qwen2.5-VL**

    Educational prototype. Outputs require clinician review.
    Use the public example dataset only.
    """)

    case_input = gr.Dropdown(
        choices=list(cases),
        value=first_case,
        label="MRI case"
    )

    slice_input = gr.Slider(
        minimum=0,
        maximum=len(volume) - 1,
        value=len(volume) // 2,
        step=1,
        label="Processed slice index"
    )

    threshold_input = gr.Slider(
        minimum=0.1,
        maximum=0.9,
        value=0.5,
        step=0.05,
        label="Segmentation threshold — not confidence"
    )

    language_input = gr.Dropdown(
        choices=["English", "Turkish"],
        value="English",
        label="Report language"
    )

    with gr.Row():
        segment_button = gr.Button("1. Segment MRI", variant="primary")
        report_button = gr.Button("2. Generate Qwen report")

    with gr.Row():
        original_output = gr.Image(label="MRI")
        overlay_output = gr.Image(label="Predicted region")

    with gr.Row():
        mask_output = gr.Image(label="Predicted mask")
        reference_output = gr.Image(label="Reference mask")

    measurements_output = gr.JSON(label="Demonstration measurements")
    report_output = gr.Textbox(label="Unverified AI draft", lines=16)

    image_outputs = [
        original_output,
        overlay_output,
        mask_output,
        reference_output,
        measurements_output
    ]

    case_input.change(
        select_case,
        inputs=case_input,
        outputs=[slice_input] + image_outputs + [report_output]
    )

    # Girdi değişirse eski sonuçları temizle.
    for control in [slice_input, threshold_input]:
        control.change(
            lambda: (None, None, None, None, None, ""),
            outputs=image_outputs + [report_output]
        )

    language_input.change(
        lambda: "",
        outputs=report_output
    )

    segment_button.click(
        segment,
        inputs=[case_input, slice_input, threshold_input],
        outputs=image_outputs
    )

    report_button.click(
        generate_report,
        inputs=[
            case_input,
            slice_input,
            threshold_input,
            language_input
        ],
        outputs=report_output
    )

demo.queue(default_concurrency_limit=1).launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c15e97c8687c9ee17.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (9 outputs left out)

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


reading all images...
preprocessing all volumes...
cropping all volumes...
padding all volumes...
resizing all volumes...
normalizing all volumes...
done creating all dataset


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
